# 06 Georgia Monte Carlo

Select **Runtime → Change runtime type → NVIDIA GPU**, then run all cells
in a fresh session. Each notebook installs its own immutable public package
checkout and verifies every input. No Python-source upload is required.

Package source: `Favioleiva/gpugwr@3368a1bc3e5f68418d4f58384673b50ebb3a4d03`. These links become usable only
after owner-authorized publication. Local tests are not Colab verification.
A failed GPU request or numerical comparison stops execution visibly.
Save the executed notebook and download the output ZIP as evidence.


In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

SOURCE_COMMIT = "3368a1bc3e5f68418d4f58384673b50ebb3a4d03"
REPOSITORY = "https://github.com/Favioleiva/gpugwr.git"
RUN = Path(tempfile.mkdtemp(prefix="gpugwr-"))
CHECKOUT = RUN / "source"
OUTPUT = RUN / "outputs"
OUTPUT.mkdir()
subprocess.run(["git", "init", str(CHECKOUT)], check=True)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "remote",
        "add",
        "origin",
        REPOSITORY,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "fetch",
        "--depth",
        "1",
        "origin",
        SOURCE_COMMIT,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "checkout",
        "--detach",
        "FETCH_HEAD",
    ],
    check=True,
)
os.environ["CUPY_CACHE_DIR"] = str(RUN / "cupy-cache")
with (OUTPUT / "bootstrap.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [
            sys.executable,
            str(CHECKOUT / "Colab/bootstrap.py"),
            "--checkout",
            str(CHECKOUT),
            "--commit",
            SOURCE_COMMIT,
            "--output",
            str(OUTPUT / "bootstrap.json"),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )
print((OUTPUT / "bootstrap.log").read_text("utf-8")[-12000:])
completed.check_returncode()
from importlib.metadata import version

stale_modules = []
for package_name in ("numpy", "scipy", "pandas", "pyproj"):
    loaded_module = sys.modules.get(package_name)
    if loaded_module is not None:
        loaded_version = getattr(loaded_module, "__version__", None)
        if loaded_version != version(package_name):
            stale_modules.append(package_name)
if stale_modules:
    raise RuntimeError(
        "Restart the Python runtime and rerun all cells; "
        "already-loaded packages changed: " + str(stale_modules)
    )


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
from gpuGWR import GWRSpatialCache, gpuGWR
from gpugwr.notebook_measurement import environment, measure
from gpugwr.reproducibility import (
    ReproductionAssets,
    export_geography,
    model_options,
    validate_arrays,
    validate_fit,
    write_json,
    zip_outputs,
)

assets = ReproductionAssets(CHECKOUT, RUN / "data-cache")
runtime = environment()
assert runtime["gpu"]["available"], (
    "A genuine GPU is required; no CPU fallback."
)
runtime["source_commit"] = SOURCE_COMMIT
write_json(OUTPUT / "environment.json", runtime)
print(json.dumps(runtime, indent=2))


## Georgia: 500 coordinate permutations

This experiment uses **unstandardized** `PctBach ~ PctFB + PctBlack + PctRural`
and an intercept, matching Contract 03 Addendum 001. It differs from the
standardized canonical fit in notebook 01. AICc bandwidth selection uses
bounds 48–159 and is repeated for every permutation of coordinates.

The statistic is population coefficient SD (`ddof=0`). For coefficient j,
`pseudo_p[j] = count(reference_SD[:, j] > observed_SD[j]) / 500`.
The comparison is strictly greater; there is **no +1 correction**.
Seed 5536 uses the legacy MT19937 ordering. The complete 500 approved
permutations, selected bandwidths, SDs and pseudo-p values are checked.


In [ ]:
frame, specification = assets.load_case("georgia")
options = model_options(specification)
options.update(standardize=False, bandwidth_bounds=(48, 159))
with threadpool_limits(limits=1):
    observed_fit, fit_timing = measure(
        lambda: gpuGWR(frame, **options, engine="gpu").fit(), "gpu"
    )
    simulation, simulation_timing = measure(
        lambda: observed_fit.spatial_variability(
            500, seed=5536, engine="gpu"
        ),
        "gpu",
    )
print(observed_fit.summary())
display(observed_fit.local_results().head())
observed_fit.export(OUTPUT / "observed_fit")
export_geography(
    observed_fit,
    frame,
    specification,
    assets,
    OUTPUT / "observed_fit",
)
simulation.export(OUTPUT / "monte_carlo")
with np.load(
    assets.acquire(assets.manifest["monte_carlo"]["reference"]),
    allow_pickle=False,
) as archive:
    reference = {name: archive[name] for name in archive.files}
actual = {name: getattr(simulation, name) for name in reference}
validation = validate_arrays(
    actual, reference, OUTPUT / "canonical_validation.json"
)
table = pd.DataFrame(
    {
        "variable": observed_fit.variable_names,
        "observed_sd": simulation.observed,
        "pseudo_p": simulation.pvalues,
    }
)
table.to_csv(OUTPUT / "pseudo_pvalues.csv", index=False)
display(table)
write_json(
    OUTPUT / "performance.json",
    {
        "fit": fit_timing,
        "simulation": simulation_timing,
        "configuration": observed_fit.configuration,
        "simulation_metadata": simulation.metadata,
    },
)
print("All 500-iteration canonical checks:", validation["passed"])


In [ ]:
archive = zip_outputs(OUTPUT)
print(f"Evidence archive: {archive}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink

    display(FileLink(str(archive)))
else:
    files.download(str(archive))
